In [0]:
from pyspark.sql.functions import col, to_timestamp, input_file_name, months_between, current_date, floor

# 1. Load Bronze - API Observations
bronze_api_path = "/FileStore/bronze/bronze_healthcare_patients"
bronze_api_df = spark.read.json(bronze_api_path)

# Add timestamp and calculate age in years
silver_api_df = (
    bronze_api_df
    .withColumn("timestamp", to_timestamp(col("birthDate")))
    .withColumn("age", floor(months_between(current_date(), col("birthDate")) / 12))
    .select(
        col("id"),
        col("gender"),
        col("birthDate"),
        col("first_name"),
        col("last_name"),
        col("age"),
        col("timestamp")
    )
)

silver_api_df.write \
                .option("mergeSchema", "true") \
                .mode("overwrite") \
                .format("delta") \
                .save("/FileStore/silver/silver_healthcare_patient")
print("✅ Silver table for healthcare API saved with age.")

from pyspark.sql.functions import input_file_name, col, when

# 2. Load Bronze - PDF Metadata
bronze_pdf_path = "/FileStore/bronze/bronze_pdfs"
bronze_pdf_df = spark.read.json(bronze_pdf_path)

# Transformation: classify PDFs by size
silver_pdf_df = (
    bronze_pdf_df
    .withColumn("source_file", input_file_name())
    .withColumn("pdf_size_category", 
                when(col("length") < 1_000_000, "small")
                .when(col("length") < 5_000_000, "medium")
                .otherwise("large"))
    .select("path", "source_file", "modificationTime", "length", "pdf_size_category")
)

silver_pdf_df.write \
                .option("mergeSchema", "true") \
                .mode("overwrite") \
                .format("delta") \
                .save("/FileStore/silver/silver_pdf_metadata")
print("✅ Silver table for PDF metadata saved with size categorization.")


✅ Silver table for healthcare API saved with age.
✅ Silver table for PDF metadata saved with size categorization.
